# Image Classification using Transfer Learning (ResNet50)

In [ ]:
# ============================================================
# Image Classification using Transfer Learning (ResNet50)
# ============================================================

import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Load CIFAR-10 Dataset
# -----------------------------
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

class_names = [
    "Airplane",
    "Automobile",
    "Bird",
    "Cat",
    "Deer",
    "Dog",
    "Frog",
    "Horse",
    "Ship",
    "Truck"
]

print("Training Images :", X_train.shape)
print("Testing Images  :", X_test.shape)

# -----------------------------
# Normalize Images
# -----------------------------
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# -----------------------------
# Resize Images
# -----------------------------
X_train = tf.image.resize(X_train, (224,224))
X_test = tf.image.resize(X_test, (224,224))

print("Resized Shape :", X_train.shape)

# -----------------------------
# Display Sample Images
# -----------------------------
plt.figure(figsize=(10,6))

for i in range(12):

    plt.subplot(3,4,i+1)

    plt.imshow(X_train[i])

    plt.title(class_names[y_train[i][0]])

    plt.axis("off")

plt.tight_layout()
plt.show()

# -----------------------------
# Load Pretrained ResNet50
# -----------------------------
base_model = keras.applications.ResNet50(

    include_top=False,

    weights="imagenet",

    input_shape=(224,224,3)

)

base_model.trainable = False

# -----------------------------
# Build Model
# -----------------------------
model = keras.Sequential([

    base_model,

    keras.layers.GlobalAveragePooling2D(),

    keras.layers.Dense(
        512,
        activation="relu"
    ),

    keras.layers.Dropout(0.5),

    keras.layers.Dense(
        10,
        activation="softmax"
    )

])

# -----------------------------
# Model Summary
# -----------------------------
model.summary()

# -----------------------------
# Compile Model
# -----------------------------
model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

# -----------------------------
# Train Model
# -----------------------------
history = model.fit(

    X_train,

    y_train,

    epochs=10,

    batch_size=32,

    validation_split=0.2,

    verbose=1

)

# -----------------------------
# Evaluate Model
# -----------------------------
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test
)

print("\nTest Loss :", test_loss)
print("Test Accuracy :", test_accuracy)

# -----------------------------
# Plot Accuracy
# -----------------------------
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Model Accuracy")
plt.legend()

# -----------------------------
# Plot Loss
# -----------------------------
plt.subplot(1,2,2)

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model Loss")
plt.legend()

plt.tight_layout()
plt.show()

# -----------------------------
# Predictions
# -----------------------------
predictions = model.predict(X_test)

# -----------------------------
# Display Predictions
# -----------------------------
plt.figure(figsize=(12,8))

for i in range(12):

    plt.subplot(3,4,i+1)

    plt.imshow(X_test[i])

    predicted = np.argmax(predictions[i])

    actual = y_test[i][0]

    color = "green" if predicted == actual else "red"

    plt.title(
        f"Pred: {class_names[predicted]}\nTrue: {class_names[actual]}",
        color=color,
        fontsize=8
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

# -----------------------------
# Single Prediction
# -----------------------------
index = 100

prediction = np.argmax(
    model.predict(
        X_test[index].numpy().reshape(1,224,224,3)
    )
)

print("Actual Class    :", class_names[y_test[index][0]])
print("Predicted Class :", class_names[prediction])